## 参考资料

中国科学院软件研究所 刘焕勇老师 https://github.com/liuhuanyong/QASystemOnMedicalKG

## 导入工具包

In [ ]:
import os
import json
from neo4j.graph import  Node

import pandas as pd

## 导入数据集csv文件

In [ ]:
df = pd.read_csv('medical_data.csv')

In [ ]:
df.shape

查看df对象结构

In [ ]:
df

## 实体（节点）

### 实体：所有症状

In [ ]:
symptoms = []
for each in df['症状']:
    symptoms.extend(each.split(','))
symptoms = set(symptoms)

In [ ]:
symptoms

### 所有科室

In [ ]:
departments = []
for each in df['科室']:
    departments.extend(each.split(','))
departments = set(departments)

### 实体：所有检查

In [ ]:
checks = []
for each in df['检查']:
    checks.extend(each.split(','))
checks = set(checks)

### 实体：所有药物

In [ ]:
drugs = []
for each in df['推荐药物']:
    try:
        drugs.extend(each.split(','))
    except:
        pass
for each in df['常用药物']:
    try:
        drugs.extend(each.split(','))
    except:
        pass
drugs = set(drugs)

### 实体：所有食物

In [ ]:
foods = []
for each in df['可以吃']:
    try:
        foods.extend(each.split(','))
    except:
        pass
for each in df['不可以吃']:
    try:
        foods.extend(each.split(','))
    except:
        pass
for each in df['推荐吃']:
    try:
        foods.extend(each.split(','))
    except:
        pass
foods = set(foods)

### 实体：所有药物厂商

In [ ]:
producers = []

for each in df['具体药物']:
    try:
        for each_drug in each.split(','):
            producer = each_drug.split('(')[0]
            producers.append(producer)
    except:
        pass
producers = set(producers)

### 疾病字典信息

In [ ]:
disease_infos = [] # 疾病信息
for idx, row in df.iterrows():
    disease_infos.append(dict(row))

In [ ]:
dict(row).keys()

## 关系（边）

In [ ]:
def deduplicate(rels_old):
    '''关系去重函数'''
    rels_new = []
    for each in rels_old:
        if each not in rels_new:
            rels_new.append(each)
    return rels_new

### 关系：疾病-检查

In [ ]:
rels_check = []
for idx, row in df.iterrows():
    for each in row['检查'].split(','):
        rels_check.append([row['疾病名称'], each])
rels_check = deduplicate(rels_check)

In [ ]:
rels_check

### 关系：疾病-症状

In [ ]:
rels_symptom = []
for idx, row in df.iterrows():
    for each in row['症状'].split(','):
        rels_symptom.append([row['疾病名称'], each])
rels_symptom = deduplicate(rels_symptom)

In [ ]:
rels_symptom

### 关系：疾病-疾病（并发症）

In [ ]:
rels_acompany = []
for idx, row in df.iterrows():
    for each in row['并发症'].split(','):
        rels_acompany.append([row['疾病名称'], each])
rels_acompany = deduplicate(rels_acompany)

In [ ]:
rels_acompany

### 关系：疾病-推荐药物

In [ ]:
rels_recommanddrug = []
for idx, row in df.iterrows():
    try:
        for each in row['推荐药物'].split(','):
            rels_recommanddrug.append([row['疾病名称'], each])
    except:
        pass
rels_recommanddrug = deduplicate(rels_recommanddrug)

### 关系：疾病-常用药物

In [ ]:
rels_commonddrug = []
for idx, row in df.iterrows():
    try:
        for each in row['常用药物'].split(','):
            rels_commonddrug.append([row['疾病名称'], each])
    except:
        pass
rels_commonddrug = deduplicate(rels_commonddrug)

### 关系：疾病-不可以吃

In [ ]:
rels_noteat = []
for idx, row in df.iterrows():
    try:
        for each in row['不可以吃'].split(','):
            rels_noteat.append([row['疾病名称'], each])
    except:
        pass
rels_noteat = deduplicate(rels_noteat)

### 关系：疾病-可以吃

In [ ]:
rels_doeat = []
for idx, row in df.iterrows():
    try:
        for each in row['可以吃'].split(','):
            rels_doeat.append([row['疾病名称'], each])
    except:
        pass
rels_doeat = deduplicate(rels_doeat)

### 关系：疾病-推荐吃

In [ ]:
rels_recommandeat = []
for idx, row in df.iterrows():
    try:
        for each in row['推荐吃'].split(','):
            rels_recommandeat.append([row['疾病名称'], each])
    except:
        pass
rels_recommandeat = deduplicate(rels_recommandeat)

### 关系：药物厂商-具体药物

In [ ]:
rels_drug_producer = []
for each in df['具体药物']:
    try:
        for each_drug in each.split(','):
            producer = each_drug.split('(')[0]
            drug = each_drug.split('(')[1][:-1]
            rels_drug_producer.append([producer, drug])
    except:
        pass
rels_drug_producer = deduplicate(rels_drug_producer)

### 关系：疾病-科室、小科室-大科室

In [ ]:
rels_category = [] # 关系：疾病-科室
rels_department = [] # 关系：小科室-大科室
for idx, row in df.iterrows():
    if len(row['科室'].split(',')) == 1:
        rels_category.append([row['疾病名称'], row['科室']])
    else:
        big = row['科室'].split(',')[0] # 大科室
        small = row['科室'].split(',')[1] # 小科室
        rels_category.append([row['疾病名称'], small])
        rels_department.append([small, big])
rels_category = deduplicate(rels_category)
rels_department = deduplicate(rels_department)

In [ ]:
rels_category

In [ ]:
rels_department

## 连接图数据库

In [ ]:
# 注意，这里的用户名为neo4j全局用户名，而非DBMS或者database的名称
from neo4j import GraphDatabase
URI = "neo4j://localhost:7687"
USERNAME = "neo4j"
PASSWORD = "12345678"
driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

## 创建知识图谱实体（节点）

### 样例代码

In [ ]:
# # Neo4j样例代码

# 创建实体
# node = Node('Disease', name='百日咳', easy_get='多见于小儿')
# g.create(node)

In [ ]:
# 删除所有实体和关系
try:
    with driver.session() as session:
        session.run("MATCH (n) DETACH DELETE n")
        print("所有节点和关系已删除")
except Exception as e:
    print(f"发生错误: {e}")

    

### 创建疾病实体

In [ ]:
# 创建疾病实体的函数
def create_disease_node(tx, disease_dict):
    query = """
    CREATE (d:Disease {
        name: $name,
        desc: $desc,
        prevent: $prevent,
        cause: $cause,
        easy_get: $easy_get,
        cure_lasttime: $cure_lasttime,
        cure_department: $cure_department,
        cure_way: $cure_way,
        cured_prob: $cured_prob
    })
    """
    tx.run(query, 
           name=disease_dict['疾病名称'],
           desc=disease_dict['疾病描述'],
           prevent=disease_dict['预防措施'],
           cause=disease_dict['病因'],
           easy_get=disease_dict['易感人群'],
           cure_lasttime=disease_dict['疗程'],
           cure_department=disease_dict['科室'],
           cure_way=disease_dict['疗法'],
           cured_prob=disease_dict['治愈率'])

# 处理 disease_infos 数据
count = 0
with driver.session() as session:
    for disease_dict in disease_infos:
        try:
            session.write_transaction(create_disease_node, disease_dict)
            count += 1
            print('创建疾病实体：', disease_dict['疾病名称'])
        except Exception as e:
            print(f"创建 {disease_dict['疾病名称']} 时出错: {e}")
            pass

print('共创建 {} 个疾病实体'.format(count))




### 创建药物实体

In [ ]:
driver

In [ ]:
# 创建药物节点
with driver.session() as session:
    for each in drugs:
        try:
            session.run("CREATE (d:Drug {name: $name})", name=each)
            print('创建实体 {}'.format(each))
        except Exception as e:
            print(f"创建 {each} 时出错: {e}")




### 创建食物实体

In [ ]:
# 创建食物节点
with driver.session() as session:
    for each in foods:
        try:
            session.run("CREATE (f:Food {name: $name})", name=each)
            print('创建实体 {}'.format(each))
        except Exception as e:
            print(f"创建 {each} 时出错: {e}")




### 创建检查实体

In [ ]:
# 创建检查节点
with driver.session() as session:
    for each in checks:
        try:
            session.run("CREATE (c:Check {name: $name})", name=each)
            print('创建实体 {}'.format(each))
        except Exception as e:
            print(f"创建 {each} 时出错: {e}")




### 创建科室实体

In [ ]:
# 创建科室节点
with driver.session() as session:
    for each in departments:
        try:
            session.run("CREATE (d:Department {name: $name})", name=each)
            print('创建实体 {}'.format(each))
        except Exception as e:
            print(f"创建 {each} 时出错: {e}")




### 创建 药物厂商 实体

In [ ]:
# 创建生产者节点
with driver.session() as session:
    for each in producers:
        try:
            session.run("CREATE (p:Producer {name: $name})", name=each)
            print('创建实体 {}'.format(each))
        except Exception as e:
            print(f"创建 {each} 时出错: {e}")




### 创建 症状 实体

In [ ]:
# 创建症状节点
with driver.session() as session:
    for each in symptoms:
        try:
            session.run("CREATE (s:Symptom {name: $name})", name=each)
            print('创建实体 {}'.format(each))
        except Exception as e:
            print(f"创建 {each} 时出错: {e}")




## 创建知识图谱关系（连接、边）

### 样例代码

In [ ]:
# # Neo4j样例代码

# # 删除所有实体和关系
# cypher = 'MATCH (n) DETACH DELETE n'
# g.run(cypher)

In [ ]:
# # 创建关系 样例代码
# start_node = 'Disease'
# end_node = 'Check'
# p = '百日咳'
# q = '血常规'
# rel_type = 'need_check'
# rel_name = '诊断检查'

# # Cypher语句
# query = "match(p:%s),(q:%s) where p.name='%s' and q.name='%s' create (p)-[rel:%s{name:'%s'}]->(q)" % (start_node, end_node, p, q, rel_type, rel_name)
# print(query)
# g.run(query) # 运行 Cypher 语句


In [ ]:
def create_relationship(start_node, end_node, edges, rel_type, rel_name):
    '''创建关系函数'''
    with driver.session() as session:
        for edge in edges:
            p = edge[0]  # 起始节点名称
            q = edge[1]  # 结束节点名称
            # 参数化的 Cypher 查询
            query = """
            MATCH (p:%s {name: $p_name}), (q:%s {name: $q_name})
            CREATE (p)-[rel:%s {name: $rel_name}]->(q)
            """ % (start_node, end_node, rel_type)
            try:
                session.run(query, p_name=p, q_name=q, rel_name=rel_name)
                print('创建关系 {}-{}->{}'.format(p, rel_type, q))
            except Exception as e:
                print(f"创建关系 {p}-{rel_type}->{q} 时出错: {e}")

### 创建所有关系

In [ ]:
create_relationship('Disease', 'Food', rels_recommandeat, 'recommand_eat', '推荐食谱')
create_relationship('Disease', 'Food', rels_noteat, 'no_eat', '忌吃')
create_relationship('Disease', 'Food', rels_doeat, 'do_eat', '宜吃')
create_relationship('Department', 'Department', rels_department, 'belongs_to', '属于')
create_relationship('Disease', 'Drug', rels_commonddrug, 'common_drug', '常用药品')
create_relationship('Producer', 'Drug', rels_drug_producer, 'drugs_of', '生产药品')
create_relationship('Disease', 'Drug', rels_recommanddrug, 'recommand_drug', '好评药品')
create_relationship('Disease', 'Check', rels_check, 'need_check', '诊断检查')
create_relationship('Disease', 'Symptom', rels_symptom, 'has_symptom', '症状')
create_relationship('Disease', 'Disease', rels_acompany, 'acompany_with', '并发症')
create_relationship('Disease', 'Department', rels_category, 'belongs_to', '所属科室')

# 关闭连接
driver.close();